# Лабораторная работа №4
## Edge ML/TinyML: классификация IMU-телеметрии и оптимизация модели

**Студенческий шаблон — TensorFlow / Keras / LiteRT**

Цель: обучить компактную модель распознавания активности по IMU, преобразовать ее в LiteRT, выполнить назначенное квантование и измерить компромисс **качество — размер — задержка**.

Максимум: **20 баллов**.

## Теория 1. Ресурсная модель TinyML

В FP32 каждый параметр занимает 4 байта, в INT8 — 1 байт:

$$
M_{FP32}=4N
$$

$$
M_{INT8}\approx N
$$

Однако размер FlatBuffer уменьшается не строго в 4 раза: в файле остаются структура графа, метаданные, смещения и другие служебные данные.

Для MCU оцениваются отдельно:

- Flash — хранение модели;
- RAM — входы, выходы, активации и tensor arena;
- latency — время вывода.

## Теория 2. Квантование

Аффинная схема:

$$
q=\mathrm{round}\left(\frac{r}{S}+Z\right)
$$

$$
r\approx S(q-Z)
$$

`S` — scale, `Z` — zero point.

**Dynamic Range:** в первую очередь уменьшается разрядность весов; вход и выход обычно остаются float.

**Full INT8:** квантуются веса и активации; требуется representative dataset и целочисленный вход/выход.

## Теория 3. Многокритериальная оптимизация

$$
F_1\rightarrow\max
$$

при ограничениях:

$$
M\le M_{max}, \qquad T\le T_{max}
$$

Инженерные показатели:

$$
Compression=\frac{Size_{FP32}}{Size_{Edge}}
$$

$$
\Delta F_1=F_{1,FP32}-F_{1,Edge}
$$

Для многоклассового распознавания в работе используется **macro-F1**.

## Шаг 1. Среда и LiteRT

Используется `ai_edge_litert.interpreter.Interpreter`, если пакет установлен. Иначе код совместимо переключается на `tf.lite.Interpreter`.

In [ ]:
# Среда выполнения
import os
import sys
import random
import time
import json
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)

tf_major, tf_minor = map(int, tf.__version__.split(".")[:2])
assert (tf_major, tf_minor) >= (2, 16), "Требуется TensorFlow 2.16+."

# LiteRT standalone runtime — предпочтительный современный Python-путь.
# Если пакет в среде отсутствует, ноутбук остается работоспособным
# через совместимый классический Interpreter из TensorFlow.
try:
    from ai_edge_litert.interpreter import Interpreter as LiteRTInterpreter
    LITERT_BACKEND = "ai-edge-litert"
except Exception as e:
    LiteRTInterpreter = tf.lite.Interpreter
    LITERT_BACKEND = "tf.lite.Interpreter (fallback)"
    print("ai-edge-litert не импортирован:", repr(e))
    print("Для standalone runtime можно выполнить:")
    print("  pip install -U ai-edge-litert")

print("Interpreter backend:", LITERT_BACKEND)


In [ ]:
def set_seed(seed=2026):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

    # По возможности включаем детерминированные TensorFlow-операции.
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


## Шаг 2. Индивидуальный вариант

Измените только `V`.

In [ ]:
VARIANTS = {
  "1": {
    "activities": [
      "WALKING",
      "SITTING",
      "STANDING"
    ],
    "architecture": "MLP",
    "quantization": "Dynamic Range",
    "model_size_kb_max": 50,
    "seed": 2027
  },
  "2": {
    "activities": [
      "WALKING",
      "WALKING_UPSTAIRS",
      "WALKING_DOWNSTAIRS"
    ],
    "architecture": "1D-CNN",
    "quantization": "Full INT8",
    "model_size_kb_max": 100,
    "seed": 2028
  },
  "3": {
    "activities": [
      "SITTING",
      "STANDING",
      "LAYING"
    ],
    "architecture": "MLP",
    "quantization": "Full INT8",
    "model_size_kb_max": 50,
    "seed": 2029
  },
  "4": {
    "activities": [
      "WALKING",
      "STANDING",
      "LAYING"
    ],
    "architecture": "1D-CNN",
    "quantization": "Dynamic Range",
    "model_size_kb_max": 100,
    "seed": 2030
  },
  "5": {
    "activities": [
      "WALKING_UPSTAIRS",
      "WALKING_DOWNSTAIRS",
      "SITTING"
    ],
    "architecture": "MLP",
    "quantization": "Dynamic Range",
    "model_size_kb_max": 50,
    "seed": 2031
  },
  "6": {
    "activities": [
      "WALKING",
      "WALKING_UPSTAIRS",
      "SITTING",
      "STANDING"
    ],
    "architecture": "1D-CNN",
    "quantization": "Full INT8",
    "model_size_kb_max": 100,
    "seed": 2032
  },
  "7": {
    "activities": [
      "WALKING",
      "WALKING_DOWNSTAIRS",
      "SITTING",
      "LAYING"
    ],
    "architecture": "MLP",
    "quantization": "Full INT8",
    "model_size_kb_max": 100,
    "seed": 2033
  },
  "8": {
    "activities": [
      "WALKING_UPSTAIRS",
      "WALKING_DOWNSTAIRS",
      "STANDING",
      "LAYING"
    ],
    "architecture": "1D-CNN",
    "quantization": "Dynamic Range",
    "model_size_kb_max": 150,
    "seed": 2034
  },
  "9": {
    "activities": [
      "WALKING",
      "SITTING",
      "LAYING"
    ],
    "architecture": "MLP",
    "quantization": "Dynamic Range",
    "model_size_kb_max": 50,
    "seed": 2035
  },
  "10": {
    "activities": [
      "WALKING",
      "WALKING_UPSTAIRS",
      "STANDING"
    ],
    "architecture": "1D-CNN",
    "quantization": "Full INT8",
    "model_size_kb_max": 100,
    "seed": 2036
  },
  "11": {
    "activities": [
      "WALKING_DOWNSTAIRS",
      "SITTING",
      "STANDING"
    ],
    "architecture": "MLP",
    "quantization": "Full INT8",
    "model_size_kb_max": 50,
    "seed": 2037
  },
  "12": {
    "activities": [
      "WALKING",
      "WALKING_UPSTAIRS",
      "WALKING_DOWNSTAIRS",
      "LAYING"
    ],
    "architecture": "1D-CNN",
    "quantization": "Dynamic Range",
    "model_size_kb_max": 150,
    "seed": 2038
  },
  "13": {
    "activities": [
      "WALKING_UPSTAIRS",
      "SITTING",
      "STANDING",
      "LAYING"
    ],
    "architecture": "MLP",
    "quantization": "Dynamic Range",
    "model_size_kb_max": 100,
    "seed": 2039
  },
  "14": {
    "activities": [
      "WALKING",
      "WALKING_DOWNSTAIRS",
      "STANDING",
      "LAYING"
    ],
    "architecture": "1D-CNN",
    "quantization": "Full INT8",
    "model_size_kb_max": 100,
    "seed": 2040
  },
  "15": {
    "activities": [
      "WALKING",
      "WALKING_UPSTAIRS",
      "SITTING"
    ],
    "architecture": "MLP",
    "quantization": "Full INT8",
    "model_size_kb_max": 50,
    "seed": 2041
  },
  "16": {
    "activities": [
      "WALKING",
      "WALKING_DOWNSTAIRS",
      "LAYING"
    ],
    "architecture": "1D-CNN",
    "quantization": "Dynamic Range",
    "model_size_kb_max": 100,
    "seed": 2042
  },
  "17": {
    "activities": [
      "WALKING_UPSTAIRS",
      "STANDING",
      "LAYING"
    ],
    "architecture": "MLP",
    "quantization": "Dynamic Range",
    "model_size_kb_max": 50,
    "seed": 2043
  },
  "18": {
    "activities": [
      "WALKING_DOWNSTAIRS",
      "SITTING",
      "LAYING"
    ],
    "architecture": "1D-CNN",
    "quantization": "Full INT8",
    "model_size_kb_max": 100,
    "seed": 2044
  },
  "19": {
    "activities": [
      "WALKING",
      "SITTING",
      "STANDING",
      "LAYING"
    ],
    "architecture": "MLP",
    "quantization": "Full INT8",
    "model_size_kb_max": 100,
    "seed": 2045
  },
  "20": {
    "activities": [
      "WALKING",
      "WALKING_UPSTAIRS",
      "WALKING_DOWNSTAIRS",
      "STANDING"
    ],
    "architecture": "1D-CNN",
    "quantization": "Dynamic Range",
    "model_size_kb_max": 150,
    "seed": 2046
  },
  "21": {
    "activities": [
      "WALKING_UPSTAIRS",
      "WALKING_DOWNSTAIRS",
      "SITTING",
      "STANDING"
    ],
    "architecture": "MLP",
    "quantization": "Dynamic Range",
    "model_size_kb_max": 100,
    "seed": 2047
  },
  "22": {
    "activities": [
      "WALKING",
      "WALKING_UPSTAIRS",
      "STANDING",
      "LAYING"
    ],
    "architecture": "1D-CNN",
    "quantization": "Full INT8",
    "model_size_kb_max": 100,
    "seed": 2048
  },
  "23": {
    "activities": [
      "WALKING",
      "WALKING_DOWNSTAIRS",
      "SITTING",
      "STANDING"
    ],
    "architecture": "MLP",
    "quantization": "Full INT8",
    "model_size_kb_max": 100,
    "seed": 2049
  },
  "24": {
    "activities": [
      "WALKING_UPSTAIRS",
      "WALKING_DOWNSTAIRS",
      "SITTING",
      "LAYING"
    ],
    "architecture": "1D-CNN",
    "quantization": "Dynamic Range",
    "model_size_kb_max": 150,
    "seed": 2050
  },
  "25": {
    "activities": [
      "WALKING",
      "WALKING_UPSTAIRS",
      "SITTING",
      "LAYING"
    ],
    "architecture": "1D-CNN",
    "quantization": "Full INT8",
    "model_size_kb_max": 100,
    "seed": 2051
  }
}

# TODO: Студент — укажите номер своего варианта.
V = 1

CFG = VARIANTS[str(V)] if str(V) in VARIANTS else VARIANTS[V]
SELECTED_ACTIVITIES = CFG["activities"]
ARCHITECTURE = CFG["architecture"]
QUANTIZATION_MODE = CFG["quantization"]
MODEL_SIZE_KB_MAX = int(CFG["model_size_kb_max"])
SEED = int(CFG["seed"])

set_seed(SEED)

print("Вариант:", V)
print("Активности:", SELECTED_ACTIVITIES)
print("Архитектура:", ARCHITECTURE)
print("Квантование:", QUANTIZATION_MODE)
print("Лимит:", MODEL_SIZE_KB_MAX, "КБ")


## Шаг 3. Загрузка UCI HAR

Используем сырые оконные сигналы `body_acc_*` и `body_gyro_*`: 128 временных отсчетов × 6 каналов.

In [ ]:
DATA_URL = (
    "https://archive.ics.uci.edu/static/public/240/"
    "human%2Bactivity%2Brecognition%2Busing%2Bsmartphones.zip"
)

DATA_DIR = Path("./data_lab04")
ZIP_PATH = DATA_DIR / "uci_har.zip"
EXTRACT_DIR = DATA_DIR / "extracted"
DATA_ROOT = EXTRACT_DIR / "UCI HAR Dataset"

DATA_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_ROOT.exists():
    if not ZIP_PATH.exists():
        print("Скачивание UCI HAR...")
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)

assert DATA_ROOT.exists(), f"Не найден каталог: {DATA_ROOT}"
print("Dataset root:", DATA_ROOT)

ACTIVITY_TO_ID = {
    "WALKING": 1,
    "WALKING_UPSTAIRS": 2,
    "WALKING_DOWNSTAIRS": 3,
    "SITTING": 4,
    "STANDING": 5,
    "LAYING": 6,
}

ACTIVITY_RU = {
    "WALKING": "Ходьба",
    "WALKING_UPSTAIRS": "Подъем по лестнице",
    "WALKING_DOWNSTAIRS": "Спуск по лестнице",
    "SITTING": "Сидение",
    "STANDING": "Стояние",
    "LAYING": "Лежание",
}

SIGNALS = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
]

def load_uci_har_split(split):
    signal_dir = DATA_ROOT / split / "Inertial Signals"

    channels = []
    for signal in SIGNALS:
        path = signal_dir / f"{signal}_{split}.txt"
        arr = np.loadtxt(path, dtype=np.float32)
        channels.append(arr)

    # [samples, 128, 6]
    X = np.stack(channels, axis=-1).astype(np.float32)

    y = np.loadtxt(
        DATA_ROOT / split / f"y_{split}.txt",
        dtype=np.int32
    )

    subject = np.loadtxt(
        DATA_ROOT / split / f"subject_{split}.txt",
        dtype=np.int32
    )

    return X, y, subject

X_train_all, y_train_all, subject_train_all = load_uci_har_split("train")
X_test_all, y_test_all, subject_test_all = load_uci_har_split("test")

print("Train raw:", X_train_all.shape, y_train_all.shape)
print("Test raw:", X_test_all.shape, y_test_all.shape)


## Шаг 4. Подвыборка и нормализация

Валидация разделяется по испытуемым. Среднее и стандартное отклонение вычисляются только по Train.

In [ ]:
def select_and_remap(X, y, subject, selected_activities):
    old_ids = [ACTIVITY_TO_ID[name] for name in selected_activities]
    mask = np.isin(y, old_ids)

    X_sub = X[mask]
    y_old = y[mask]
    subject_sub = subject[mask]

    old_to_new = {old_id: i for i, old_id in enumerate(old_ids)}
    y_new = np.array([old_to_new[int(v)] for v in y_old], dtype=np.int32)

    return X_sub, y_new, subject_sub

X_train_pool, y_train_pool, subject_train_pool = select_and_remap(
    X_train_all, y_train_all, subject_train_all, SELECTED_ACTIVITIES
)

X_test, y_test, subject_test = select_and_remap(
    X_test_all, y_test_all, subject_test_all, SELECTED_ACTIVITIES
)

# Валидация выделяется по людям, а не по случайным окнам.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.18,
    random_state=SEED
)

train_idx, val_idx = next(
    splitter.split(
        X_train_pool,
        y_train_pool,
        groups=subject_train_pool
    )
)

X_train = X_train_pool[train_idx]
y_train = y_train_pool[train_idx]

X_val = X_train_pool[val_idx]
y_val = y_train_pool[val_idx]

# Нормализация — только по Train.
channel_mean = X_train.mean(axis=(0, 1), keepdims=True)
channel_std = X_train.std(axis=(0, 1), keepdims=True) + 1e-6

X_train = ((X_train - channel_mean) / channel_std).astype(np.float32)
X_val = ((X_val - channel_mean) / channel_std).astype(np.float32)
X_test = ((X_test - channel_mean) / channel_std).astype(np.float32)

NUM_CLASSES = len(SELECTED_ACTIVITIES)
INPUT_SHAPE = X_train.shape[1:]

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)
print("Subjects train:", np.unique(subject_train_pool[train_idx]))
print("Subjects val:", np.unique(subject_train_pool[val_idx]))
print("Input shape:", INPUT_SHAPE)
print("Classes:", SELECTED_ACTIVITIES)

# Параметры нормализации пригодятся при реальном развертывании.
np.savez(
    "normalization_params.npz",
    mean=channel_mean,
    std=channel_std,
    signals=np.array(SIGNALS)
)


In [ ]:
def plot_window(x, label, class_names):
    t = np.arange(x.shape[0]) / 50.0

    fig = plt.figure(figsize=(10, 5))
    for ch in range(3):
        plt.plot(t, x[:, ch], label=SIGNALS[ch])

    plt.xlabel("Время, с")
    plt.ylabel("Нормализованное ускорение")
    plt.title(f"Пример окна: {class_names[label]}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

plot_window(X_train[0], y_train[0], SELECTED_ACTIVITIES)


## Шаг 5. Архитектура

Реализуйте архитектуру своего варианта. Последний слой возвращает **логиты**.

In [ ]:
def build_mlp(input_shape, num_classes):
    # TODO: Студент.
    # Рекомендуемая архитектура:
    # Input -> Flatten -> Dense(32, relu) -> Dropout(0.2)
    # -> Dense(16, relu) -> Dense(num_classes)
    raise NotImplementedError("Реализуйте узкий MLP.")


def build_cnn(input_shape, num_classes):
    # TODO: Студент.
    # Рекомендуемая архитектура:
    # Input
    # -> Conv1D(64, 5, strides=2, relu, padding='same')
    # -> Conv1D(96, 3, strides=2, relu, padding='same')
    # -> Conv1D(128, 3, strides=2, relu, padding='same')
    # -> GlobalAveragePooling1D
    # -> Dense(64, relu)
    # -> Dropout(0.2)
    # -> Dense(num_classes)
    raise NotImplementedError("Реализуйте 1D-CNN.")


# TODO: Студент — выберите функцию по ARCHITECTURE.
# if ARCHITECTURE == "MLP":
#     model = build_mlp(INPUT_SHAPE, NUM_CLASSES)
# else:
#     model = build_cnn(INPUT_SHAPE, NUM_CLASSES)


## Шаг 6. FP32-обучение

Используйте Adam и `SparseCategoricalCrossentropy(from_logits=True)`.

In [ ]:
# TODO: Студент — скомпилируйте и обучите модель.

# model.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=1e-3),
#     loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
#     metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
# )
#
# callbacks = [
#     keras.callbacks.EarlyStopping(
#         monitor="val_loss",
#         patience=3,
#         restore_best_weights=True
#     )
# ]
#
# history = model.fit(
#     X_train, y_train,
#     validation_data=(X_val, y_val),
#     epochs=15,
#     batch_size=128,
#     callbacks=callbacks,
#     verbose=2,
# )


In [ ]:
def plot_history(history):
    hist = history.history

    plt.figure(figsize=(7, 4))
    plt.plot(hist["loss"], label="Train")
    plt.plot(hist["val_loss"], label="Validation")
    plt.xlabel("Эпоха")
    plt.ylabel("Loss")
    plt.title("Функция потерь")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(hist["accuracy"], label="Train")
    plt.plot(hist["val_accuracy"], label="Validation")
    plt.xlabel("Эпоха")
    plt.ylabel("Accuracy")
    plt.title("Точность")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


## Шаг 7. Сохранение исходной модели

Сохраните `.keras`. Отдельно далее будет создан FP32 LiteRT FlatBuffer для честного расчета Compression.

In [ ]:
# TODO: Студент — сохраните исходную модель и создайте FP32 LiteRT.

# ARTIFACT_DIR = Path("./lab04_artifacts")
# ARTIFACT_DIR.mkdir(exist_ok=True)
#
# KERAS_PATH = ARTIFACT_DIR / f"variant_{V:02d}_fp32.keras"
# model.save(KERAS_PATH)
#
# keras_size_bytes = KERAS_PATH.stat().st_size
#
# # Для корректного Compression сравниваем LiteRT FlatBuffer с LiteRT FlatBuffer.
# fp32_converter = tf.lite.TFLiteConverter.from_keras_model(model)
# fp32_litert = fp32_converter.convert()
#
# FP32_LITERT_PATH = ARTIFACT_DIR / f"variant_{V:02d}_fp32.tflite"
# FP32_LITERT_PATH.write_bytes(fp32_litert)
#
# fp32_litert_size_bytes = FP32_LITERT_PATH.stat().st_size
#
# print("Keras size:", keras_size_bytes / 1024, "KiB")
# print("FP32 LiteRT size:", fp32_litert_size_bytes / 1024, "KiB")


In [ ]:
def keras_metrics(model, X, y):
    logits = model.predict(X, batch_size=256, verbose=0)
    pred = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(y, pred),
        "f1_macro": f1_score(y, pred, average="macro"),
        "y_pred": pred,
    }


## Шаг 8. Конвертация и Post-Training Quantization

Функции конвертации готовы. Студент выбирает режим согласно варианту.

In [ ]:
def representative_dataset(X, n_samples=300, seed=2026):
    rng = np.random.default_rng(seed)
    n = min(n_samples, len(X))
    indices = rng.choice(len(X), size=n, replace=False)

    def generator():
        for idx in indices:
            sample = X[idx:idx + 1].astype(np.float32)
            yield [sample]

    return generator


def convert_dynamic_range(model):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    return converter.convert()


def convert_full_int8(model, X_calibration, seed=2026):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset(
        X_calibration,
        n_samples=300,
        seed=seed,
    )

    # Требуем только INT8-совместимые builtin-операции.
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS_INT8
    ]

    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    return converter.convert()


In [ ]:
# TODO: Студент — выполните режим, назначенный вариантом.

# if QUANTIZATION_MODE == "Dynamic Range":
#     edge_model = convert_dynamic_range(model)
# else:
#     edge_model = convert_full_int8(model, X_train, seed=SEED)
#
# EDGE_PATH = ARTIFACT_DIR / f"variant_{V:02d}_edge.tflite"
# EDGE_PATH.write_bytes(edge_model)
#
# edge_size_bytes = EDGE_PATH.stat().st_size
# print("Edge LiteRT size:", edge_size_bytes / 1024, "KiB")


## Шаг 9. LiteRT Interpreter

Готовые функции автоматически учитывают тип входа `float32`/`int8` и scale/zero point.

In [ ]:
def make_interpreter(model_path, num_threads=1):
    # ai-edge-litert сохраняет классический Interpreter API.
    try:
        interpreter = LiteRTInterpreter(
            model_path=str(model_path),
            num_threads=num_threads
        )
    except TypeError:
        interpreter = LiteRTInterpreter(model_path=str(model_path))

    interpreter.allocate_tensors()
    return interpreter


def tensor_quant_params(detail):
    qp = detail.get("quantization_parameters", {})
    scales = qp.get("scales", np.array([], dtype=np.float32))
    zero_points = qp.get("zero_points", np.array([], dtype=np.int32))

    if len(scales) == 0:
        return None, None

    return float(scales[0]), int(zero_points[0])


def prepare_input(sample, detail):
    dtype = np.dtype(detail["dtype"])
    sample = sample.astype(np.float32)

    if np.issubdtype(dtype, np.integer):
        scale, zero_point = tensor_quant_params(detail)
        if scale is None or scale == 0:
            raise ValueError("Не найдены параметры квантования входа.")

        q = np.round(sample / scale + zero_point)
        info = np.iinfo(dtype)
        q = np.clip(q, info.min, info.max)
        return q.astype(dtype)

    return sample.astype(dtype)


def dequantize_output(output, detail):
    dtype = np.dtype(detail["dtype"])

    if np.issubdtype(dtype, np.integer):
        scale, zero_point = tensor_quant_params(detail)
        if scale is None or scale == 0:
            raise ValueError("Не найдены параметры квантования выхода.")

        return scale * (
            output.astype(np.float32) - zero_point
        )

    return output.astype(np.float32)


def predict_litert(model_path, X):
    interpreter = make_interpreter(model_path)

    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]

    predictions = np.empty(len(X), dtype=np.int32)

    for i in range(len(X)):
        sample = X[i:i + 1]
        input_data = prepare_input(sample, input_detail)

        interpreter.set_tensor(input_detail["index"], input_data)
        interpreter.invoke()

        output = interpreter.get_tensor(output_detail["index"])
        output = dequantize_output(output, output_detail)

        predictions[i] = int(np.argmax(output[0]))

    return predictions


def litert_metrics(model_path, X, y):
    pred = predict_litert(model_path, X)

    return {
        "accuracy": accuracy_score(y, pred),
        "f1_macro": f1_score(y, pred, average="macro"),
        "y_pred": pred,
    }


## Шаг 10. Benchmark 1000 окон и RAM-прокси

Измеряется только `invoke()`. Это benchmark среды Colab, а не окончательная latency микроконтроллера.

In [ ]:
def benchmark_litert(
    model_path,
    X,
    n_samples=1000,
    warmup=50,
    seed=2026
):
    """
    Измеряем только interpreter.invoke().
    set_tensor, квантизация входа и get_tensor в latency не входят.
    """
    interpreter = make_interpreter(model_path, num_threads=1)

    input_detail = interpreter.get_input_details()[0]

    rng = np.random.default_rng(seed)
    n = min(n_samples, len(X))
    indices = rng.choice(len(X), size=n, replace=False)

    prepared = [
        prepare_input(X[i:i + 1], input_detail)
        for i in indices
    ]

    # Warm-up
    for i in range(min(warmup, len(prepared))):
        interpreter.set_tensor(
            input_detail["index"],
            prepared[i]
        )
        interpreter.invoke()

    elapsed_us = np.empty(len(prepared), dtype=np.float64)

    for i, input_data in enumerate(prepared):
        interpreter.set_tensor(input_detail["index"], input_data)

        t0 = time.perf_counter_ns()
        interpreter.invoke()
        t1 = time.perf_counter_ns()

        elapsed_us[i] = (t1 - t0) / 1000.0

    return {
        "n": len(elapsed_us),
        "mean_us": float(np.mean(elapsed_us)),
        "median_us": float(np.median(elapsed_us)),
        "p95_us": float(np.percentile(elapsed_us, 95)),
    }


def tensor_memory_proxy_bytes(model_path):
    """
    Суммируем размеры объявленных тензоров интерпретатора.
    Это учебный RAM-прокси, а не точный peak tensor arena на MCU.
    """
    interpreter = make_interpreter(model_path, num_threads=1)

    total = 0
    for detail in interpreter.get_tensor_details():
        shape = np.asarray(detail.get("shape", []), dtype=np.int64)

        if shape.size == 0 or np.any(shape < 0):
            continue

        try:
            itemsize = np.dtype(detail["dtype"]).itemsize
        except Exception:
            continue

        total += int(np.prod(shape)) * itemsize

    return total


In [ ]:
# TODO: Студент — оцените FP32 LiteRT и Edge LiteRT.

# fp32_edge_metrics = litert_metrics(FP32_LITERT_PATH, X_test, y_test)
# edge_metrics = litert_metrics(EDGE_PATH, X_test, y_test)
#
# fp32_bench = benchmark_litert(
#     FP32_LITERT_PATH, X_test, n_samples=1000, seed=SEED
# )
# edge_bench = benchmark_litert(
#     EDGE_PATH, X_test, n_samples=1000, seed=SEED
# )
#
# fp32_ram_proxy = tensor_memory_proxy_bytes(FP32_LITERT_PATH)
# edge_ram_proxy = tensor_memory_proxy_bytes(EDGE_PATH)


## Шаг 11. Матрица ошибок Edge-модели

In [ ]:
def show_confusion(y_true, y_pred, class_names, title):
    cm = confusion_matrix(y_true, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names,
    )

    fig, ax = plt.subplots(figsize=(8, 7))
    disp.plot(
        ax=ax,
        cmap="Blues",
        values_format="d",
        colorbar=False,
    )
    plt.title(title)
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()

    return cm


### TODO: Студент

После получения `edge_metrics`:

```python
show_confusion(
    y_test,
    edge_metrics["y_pred"],
    [ACTIVITY_RU[x] for x in SELECTED_ACTIVITIES],
    "Edge LiteRT: матрица ошибок"
)
```

Проанализируйте минимум две пары ошибок.

## Шаг 12. Compression и Delta F1

In [ ]:
# TODO: Студент — рассчитайте инженерные метрики.

# compression = fp32_litert_size_bytes / edge_size_bytes
# delta_f1 = (
#     fp32_edge_metrics["f1_macro"]
#     - edge_metrics["f1_macro"]
# )
#
# size_ok = edge_size_bytes <= MODEL_SIZE_KB_MAX * 1024
#
# print("Compression:", compression)
# print("Delta F1:", delta_f1)
# print("Size OK:", size_ok)


## Итоговый отчет студента

Заполните таблицу фактическими результатами.

| Показатель | FP32 LiteRT | Edge LiteRT |
|---|---:|---:|
| Размер, KiB | ... | ... |
| Macro-F1 | ... | ... |
| Median latency, µs | ... | ... |
| P95 latency, µs | ... | ... |
| RAM-прокси, KiB | ... | ... |

Дополнительно:

- `Compression = ...`;
- `Delta F1 = ...`;
- лимит размера выполнен / не выполнен;
- педагогический/робототехнический сценарий;
- ограничения benchmark в Colab;
- окончательный вывод о целесообразности переноса на MCU.

## БРС — 20 баллов

| Элемент | Баллы |
|---|---:|
| FP32-модель | 3 |
| Предобработка IMU | 3 |
| LiteRT и квантование | 4 |
| Размер и latency | 3 |
| Compression и Delta F1 | 3 |
| Анализ компромисса | 2 |
| Edge-интерпретация | 1 |
| Воспроизводимость | 1 |
| **Итого** | **20** |